# E1 — Cross-modal convergence: an image-only encoder vs a text-only encoder

**The question.** Experiments A and B compared models within one modality,
where row *i* was literally the same input. This asks the harder version:
do a vision model that has never seen text and a text model that has never
seen an image build representations related by one linear map? Row *i* here
is an image and *a caption describing it* — two different objects sharing a
referent. This is the Platonic Representation Hypothesis's actual claim.

**Model choice matters.** The image encoder must be self-supervised
(DINOv2, MAE) — **never CLIP-family**, or the text alignment being tested
was trained in, and the experiment is circular. Same for the text side:
a plain text encoder, not a CLIP text tower.

**Pre-registered bands — fixed before any result is seen.**

| Retrieval R@1, as % of the SigLIP ceiling (0.630) | Verdict |
|---|---|
| > 50% | strong cross-modal linear correspondence at this scale |
| 15–50% | measurable but weak — consistent with PRH's scale-dependence |
| 2–15% | marginal; report as a boundary, not a positive result |
| < 2% (chance is 1/1000 = 0.1%) | no linear cross-modal correspondence at this scale |

The middle bands are the honest expectation: PRH predicts alignment grows
with model capability, and these are modest models. A weak-but-above-chance
result is a legitimate finding, not a failure — and repeating with a larger
image encoder would turn this project's two-point sample into a slope.

**Power gate.** The governing ratio is training rows / image-encoder width.
Below 5 the fit is starved and biases *downward* — exactly the run-1 failure
that would fake a negative here. The notebook refuses to interpret results
below that threshold.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
!pip -q install torch torchvision transformers sentence-transformers pillow

In [ ]:
import numpy as np, torch, json
from pathlib import Path
DATA_DIR = Path(os.environ["DATA_DIR"]); DATA_DIR.mkdir(exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.default_rng(0)

# ---- configuration ----
IMG_MODEL  = "facebook/dinov2-base"        # self-supervised, never saw text
TXT_MODEL  = "BAAI/bge-m3"                 # text-only encoder
N_PAIRS    = 8000                         # 7000/768 = 9.1 rows per image dim
N_EVAL     = 1000                          # gallery size, matches Exp B
ALPHA      = 1e-2
CEILING_R1 = 0.630                         # SigLIP jointly-trained ceiling
print("device:", DEV)

## 1. Data — COCO train2017 captions

Uses the official annotation zip (no loader scripts). Images stream from
their URLs; only embeddings are kept, never the pixels.

In [ ]:
import zipfile, urllib.request, io
ANN = DATA_DIR / "annotations_trainval2017.zip"
if not ANN.exists():
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/annotations/"
        "annotations_trainval2017.zip", str(ANN))
with zipfile.ZipFile(ANN) as z:
    with z.open("annotations/captions_train2017.json") as f:
        ann = json.load(f)

ALL_CAPTIONS = True        # True: average all 5 caption embeddings
                           # (less annotator noise); False: first only
caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
first_cap = {k: (v if ALL_CAPTIONS else v[:1]) for k, v in caps.items()}
url = {im["id"]: im["coco_url"] for im in ann["images"]}
ids = sorted(set(first_cap) & set(url))[:N_PAIRS]
captions = [first_cap[i] for i in ids]   # list of lists
print(f"{len(ids)} image/caption pairs")
print("power check: rows/768 =", f"{0.75*len(ids)/768:.1f}",
      "(need >= 5)")

## 2. Encode — image side (DINOv2), parallel fetch

The images are downloaded from COCO URLs, and that download is the whole
cost of this notebook: sequential fetching leaves the GPU idle >95% of the
time (measured: ~1.5 h for 20k images). This cell fetches with 32 worker
threads, runs the encoder in fp16, and checkpoints every 512 images to
`e1_img_ckpt.npz` — so a Colab disconnect resumes instead of restarting.

Expect a few minutes rather than an hour. Live throughput and ETA are
printed; if a single fetch is slow, raise `WORKERS` to 64.

In [ ]:
import io, urllib.request, numpy as np, torch
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoImageProcessor, AutoModel
from PIL import Image

WORKERS   = 32          # parallel downloads - the whole speedup
CHUNK     = 512         # images fetched per round
BATCH     = 64          # GPU batch
CKPT      = DATA_DIR / "e1_img_ckpt.npz"

proc = AutoImageProcessor.from_pretrained(IMG_MODEL)
vis  = AutoModel.from_pretrained(IMG_MODEL).to(DEV).eval()
if DEV == "cuda":
    vis = vis.half()                      # fp16: ~2x on L4

def fetch(n):
    """returns (row_index, PIL image) or (row_index, None)"""
    try:
        with urllib.request.urlopen(url[ids[n]], timeout=8) as r:
            return n, Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception:
        return n, None

# resume if a previous run was interrupted
if CKPT.exists():
    d = np.load(str(CKPT))
    img_list = [d["img"]]; keep = list(d["keep"]); start = int(d["next"])
    print(f"resuming from row {start} ({len(keep)} already encoded)")
else:
    img_list, keep, start = [], [], 0

import time
t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for c0 in range(start, len(ids), CHUNK):
        c1 = min(c0 + CHUNK, len(ids))
        got = sorted((r for r in pool.map(fetch, range(c0, c1))
                      if r[1] is not None), key=lambda x: x[0])
        for b0 in range(0, len(got), BATCH):
            batch = got[b0:b0 + BATCH]
            with torch.no_grad():
                x = proc(images=[im for _, im in batch],
                         return_tensors="pt").to(DEV)
                if DEV == "cuda":
                    x["pixel_values"] = x["pixel_values"].half()
                h = vis(**x).last_hidden_state[:, 0]
            img_list.append(h.float().cpu().numpy())
            keep += [n for n, _ in batch]
        done = c1 - start
        rate = done / max(time.time() - t0, 1e-9)
        eta = (len(ids) - c1) / max(rate, 1e-9) / 60
        print(f"  {c1}/{len(ids)}  kept {len(keep)}  "
              f"{rate:.0f} img/s  ETA {eta:.1f} min")
        np.savez_compressed(str(CKPT),
                            img=np.concatenate(img_list).astype(np.float32),
                            keep=np.array(keep), next=c1)

IMG = np.concatenate(img_list).astype(np.float64)
print("image embeddings:", IMG.shape,
      f"in {(time.time()-t0)/60:.1f} min")


In [ ]:
from sentence_transformers import SentenceTransformer
txt_model = SentenceTransformer(TXT_MODEL, device=DEV)

# average all captions per image: the target becomes "what the image is
# about" rather than "what one annotator happened to write"
flat, owner = [], []
for j, k in enumerate(keep):
    for cap in captions[k]:
        flat.append(cap); owner.append(j)
E = txt_model.encode(flat, batch_size=128, show_progress_bar=True,
                     convert_to_numpy=True).astype(np.float64)
owner = np.array(owner)
TXT = np.zeros((len(keep), E.shape[1]))
for j in range(len(keep)):
    TXT[j] = E[owner == j].mean(0)
print(f"{len(flat)} captions -> {TXT.shape} averaged targets "
      f"({len(flat)/len(keep):.1f} per image)")

assert len(IMG) == len(TXT), "row alignment broken"
np.savez_compressed(str(DATA_DIR / "crossmodal_pairs.npz"),
                    img=IMG.astype(np.float32), txt=TXT.astype(np.float32))
print("saved crossmodal_pairs.npz")

## 3. The power gate — refuse to interpret an underpowered result

In [ ]:
d_img = IMG.shape[1]
idx = rng.permutation(len(IMG))
te, tr = idx[:N_EVAL], idx[N_EVAL:]
ratio = len(tr) / d_img
print(f"training rows {len(tr)} / image dim {d_img} = {ratio:.1f}")
POWERED = ratio >= 5
print("POWERED" if POWERED else
      "UNDERPOWERED - a negative result here is NOT interpretable; "
      "increase N_PAIRS before drawing any conclusion")

In [ ]:
# shared helpers - defined once, used by the ceiling cell and the evaluation
def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)

def recall(S):
    order = np.argsort(-S, axis=1)
    r = (order == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

print("helpers ready: l2n, ridge, recall")

## 3b. Measure the ceiling for THIS direction — and on the SAME task

`CEILING_R1 = 0.630` was measured in Experiment B for **text -> image**.
E1 runs **image -> text**, so the borrowed figure is only approximate.

Two things must match between the system and its ceiling, or the
percentage is meaningless:

1. **Direction** — both must be image -> text.
2. **Caption treatment** — if the system retrieves against *averaged*
   caption embeddings (`ALL_CAPTIONS = True`), the ceiling must too.
   Averaging produces a cleaner, more canonical target per image, which
   makes retrieval easier; scoring the system on averaged captions against
   a ceiling measured on single captions inflates the percentage.

This cell measures SigLIP's own two towers on the same held-out rows, with
the same caption treatment.

In [ ]:
# exact ceiling: SigLIP image tower -> SigLIP text tower, same eval rows,
# and the SAME caption treatment as the system (averaged when ALL_CAPTIONS)
from transformers import AutoProcessor, AutoModel as HFAutoModel
from concurrent.futures import ThreadPoolExecutor

SIG = "google/siglip-base-patch16-224"
sp = AutoProcessor.from_pretrained(SIG)
sm = HFAutoModel.from_pretrained(SIG).to(DEV).eval()

def _tensor(o):
    """transformers versions differ: tensor or output object."""
    if torch.is_tensor(o):
        return o
    for a in ("image_embeds", "text_embeds", "pooler_output"):
        v = getattr(o, a, None)
        if v is not None:
            return v
    v = getattr(o, "last_hidden_state", None)
    if v is not None:
        return v.mean(1)
    raise TypeError(f"cannot extract a tensor from {type(o)}")

def _img(n):
    r = fetch(n)
    return r[1] if isinstance(r, tuple) else r

def _caps(n):
    """ALL captions for image n - must match how TXT was built."""
    c = captions[n]
    return c if isinstance(c, list) else [c]

def _sig_text(strings, bs=64):
    out = []
    for b in range(0, len(strings), bs):
        with torch.no_grad():
            y = sp(text=strings[b:b + bs], padding="max_length",
                   truncation=True, return_tensors="pt").to(DEV)
            try:
                v = sm.get_text_features(**y)
            except Exception:
                v = sm(input_ids=y["input_ids"]).text_embeds
            out.append(_tensor(v).float().cpu().numpy())
    return np.concatenate(out)

def _sig_image(images, bs=32):
    out = []
    for b in range(0, len(images), bs):
        with torch.no_grad():
            x = sp(images=images[b:b + bs], return_tensors="pt").to(DEV)
            try:
                v = sm.get_image_features(**x)
            except Exception:
                v = sm(pixel_values=x["pixel_values"]).image_embeds
            out.append(_tensor(v).float().cpu().numpy())
    return np.concatenate(out)

def siglip_ceiling(rows):
    want = [int(keep[r]) for r in rows]
    with ThreadPoolExecutor(max_workers=32) as pool:
        got = list(pool.map(_img, want))
    ok = [(n, im) for n, im in zip(want, got) if im is not None]
    if len(ok) < 100:
        print(f"only {len(ok)} images available - ceiling not measured")
        return None, 0, 0

    I = l2n(_sig_image([im for _, im in ok]))

    # text side, matching the system's treatment exactly
    flat, owner = [], []
    for j, (n, _) in enumerate(ok):
        for cap in _caps(n):
            flat.append(cap); owner.append(j)
    owner = np.array(owner)
    E = _sig_text(flat)
    T = np.stack([E[owner == j].mean(0) for j in range(len(ok))])
    T = l2n(T)
    per_img = len(flat) / len(ok)
    return recall(I @ T.T), len(ok), per_img

ceil, n_gal, per_img = siglip_ceiling(te)
if ceil:
    CEILING_R1 = ceil[1]
    print(f"MEASURED image->text ceiling (SigLIP, gallery {n_gal}, "
          f"{per_img:.1f} captions/image averaged):")
    print("   " + "  ".join(f"R@{k}={v:.3f}" for k, v in ceil.items()))
    print("   caption treatment now MATCHES the system - the system and")
    print("   the ceiling are scored on the same task")
    print("   (Experiment B's borrowed text->image figure was 0.630;")
    print("    a single-caption ceiling measured earlier gave 0.648)")
    if n_gal < len(te):
        print(f"   note: gallery {n_gal}, not {len(te)} - R@K depends on")
        print("   gallery size, compare only against this count")
else:
    print("keeping the borrowed ceiling 0.630")

## 4. Fit and evaluate, with all four controls

1. **shuffle control** — misaligned rows; the honest fit must beat it
2. **random-weights image encoder** — architecture-only baseline
3. **raw cross-space** — the chance floor
4. **same-modality reference** — this project's own 0.592 / 0.737

In [ ]:
W = ridge(IMG[tr], TXT[tr])
P = IMG[te] @ W
r2 = 1 - ((TXT[te]-P)**2).sum() / ((TXT[te]-TXT[te].mean(0))**2).sum()
cos = float((l2n(P) * l2n(TXT[te])).sum(1).mean())

Ws = ridge(IMG[tr], TXT[tr][rng.permutation(len(tr))])
r2s = 1 - ((TXT[te]-IMG[te]@Ws)**2).sum() / \
          ((TXT[te]-TXT[te].mean(0))**2).sum()

gal = l2n(TXT[te])
r_adapt = recall(l2n(P) @ gal.T)
raw = np.zeros_like(TXT[te]); m = min(d_img, TXT.shape[1])
raw[:, :m] = IMG[te][:, :m]
r_raw = recall(l2n(raw) @ gal.T)

print(f"held-out ridge : R2 {r2:.3f}   cosine {cos:.3f}")
print(f"shuffle control: R2 {r2s:.3f}   (gap must exceed 0.2)")
print(f"retrieval image->text: " +
      "  ".join(f"R@{k}={v:.3f}" for k, v in r_adapt.items()))
print(f"raw (chance floor)   : " +
      "  ".join(f"R@{k}={v:.3f}" for k, v in r_raw.items()))
pct = 100 * r_adapt[1] / CEILING_R1
print(f"\nR@1 as % of the SigLIP ceiling ({CEILING_R1}): {pct:.1f}%")
print(f"same-modality reference from this project: "
      f"R2 0.592 (vision pair), 0.737 (LLM pair)")

if not POWERED:
    print("\nVERDICT WITHHELD - underpowered")
elif r2 - r2s < 0.2:
    print("\nVERDICT: alignment check FAILED - fit does not beat the "
          "shuffle control; suspect row misalignment before anything else")
elif pct > 50:
    print("\nVERDICT: STRONG cross-modal linear correspondence")
elif pct > 15:
    print("\nVERDICT: MEASURABLE BUT WEAK - consistent with PRH's "
          "scale-dependence. Repeat with a larger image encoder to get a "
          "slope rather than a point.")
elif pct > 2:
    print("\nVERDICT: MARGINAL - report as a boundary, not a positive")
else:
    print("\nVERDICT: NO linear cross-modal correspondence at this scale")

## 5. Optional — the scale slope (the point of the whole exercise)

Re-run cells 2–4 with a larger image encoder (`facebook/dinov2-large`,
then `dinov2-giant`) keeping everything else fixed, and plot R@1 against
encoder size. PRH predicts the trend rises. Two points make a line; three
make it credible — and a slope is exactly what this project's stated
limitation ('two small models are a two-point sample') asks for.

In [ ]:
# collect results across encoder sizes, then:
# sizes = {"dinov2-small": r1_s, "dinov2-base": r1_b, "dinov2-large": r1_l}
# plot and report the trend; a rising line is direct PRH evidence,
# a flat line at this scale is an equally publishable negative.
print("run the cells above per encoder, record R@1, then plot")